# UpLift — Notebook 03: Model Training & Evaluation
**Project:** UpLift — Predictive Maintenance for Transit Accessibility Equipment  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/uplift-transit](https://github.com/meyeringn/uplift-transit)

---

## What This Notebook Does

1. Trains an **XGBoost** binary classifier on the SMOTE-balanced training data
2. Applies the **0.30 decision threshold** (not 0.50) — calibrated for asymmetric error costs
3. Evaluates on the locked holdout set using **F2-Score**, AUC-ROC, and Precision-Recall curves
4. Generates a **SHAP feature importance** plot — the key explainability output for transit agencies
5. Produces the **ranked equipment risk list** — the operational output agencies would use

### The Core Asymmetry

| Error | Consequence | Cost |
|---|---|---|
| **False Positive** | Dispatched a tech — equipment was fine | ~\$500–\$2,000. Recoverable. |
| **False Negative** | Missed a failure — Disabled rider stranded | ADA enforcement. Reputational. Settlement risk. **Not recoverable.** |

> A 0.30 threshold flags more equipment as at-risk, accepting more false positives in exchange for catching more real failures. This is the right trade-off when the people paying the cost of misses are Disabled riders.

In [ ]:
# Install dependencies (run once in Google Colab)
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, fbeta_score, average_precision_score
)

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Load Preprocessed Data

In [ ]:
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_holdout = np.load('X_holdout.npy')
y_holdout = np.load('y_holdout.npy')
feature_cols = joblib.load('uplift_feature_cols.pkl')

print(f'Training: {X_train.shape}  |  Holdout: {X_holdout.shape}')
print(f'Features: {feature_cols}')

## 2. Train XGBoost Model

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,   # Class weighting handled by SMOTE upstream
    eval_metric='logloss',
    random_state=42,
    use_label_encoder=False
)

model.fit(
    X_train, y_train,
    eval_set=[(X_holdout, y_holdout)],
    verbose=50
)

joblib.dump(model, 'uplift_xgboost_model.pkl')
print('\nModel saved: uplift_xgboost_model.pkl')

## 3. Evaluate at Decision Threshold = 0.30

In [ ]:
THRESHOLD = 0.30

y_prob = model.predict_proba(X_holdout)[:, 1]
y_pred = (y_prob >= THRESHOLD).astype(int)

auc_roc = roc_auc_score(y_holdout, y_prob)
f2 = fbeta_score(y_holdout, y_pred, beta=2)
avg_precision = average_precision_score(y_holdout, y_prob)

print(f'=== UpLift Model — Holdout Evaluation ===')
print(f'Decision Threshold: {THRESHOLD}')
print(f'AUC-ROC:            {auc_roc:.4f}')
print(f'F2-Score:           {f2:.4f}  ← Primary metric (recall-weighted)')
print(f'Avg Precision:      {avg_precision:.4f}')
print()
print(classification_report(y_holdout, y_pred, 
      target_names=['No Outage', 'Outage']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_holdout, y_pred)
labels = np.array([
    [f'True Negative\n{cm[0,0]}\n(Correctly safe)', f'False Positive\n{cm[0,1]}\n(Wasted service call)'],
    [f'False Negative\n{cm[1,0]}\n(⚠ Rider stranded)', f'True Positive\n{cm[1,1]}\n(Caught before failure)']
])
sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', ax=ax,
            xticklabels=['Predicted: Safe', 'Predicted: At Risk'],
            yticklabels=['Actual: Safe', 'Actual: At Risk'],
            linewidths=2, linecolor='white')
ax.set_title(f'UpLift Confusion Matrix\nThreshold = {THRESHOLD} | F2-Score = {f2:.3f}',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('uplift_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_confusion_matrix.png')

## 4. Precision-Recall & ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('UpLift — Model Performance Curves', fontweight='bold', fontsize=14)

# Precision-Recall Curve
prec, rec, pr_thresholds = precision_recall_curve(y_holdout, y_prob)
axes[0].plot(rec, prec, color='#1565C0', lw=2.5, label=f'AP = {avg_precision:.3f}')
axes[0].axhline(y=sum(y_holdout)/len(y_holdout), color='gray', 
                linestyle='--', alpha=0.7, label='Baseline (random)')
# Mark the operating threshold
idx = np.argmin(np.abs(pr_thresholds - THRESHOLD))
axes[0].scatter(rec[idx], prec[idx], s=150, color='#E53935', zorder=5,
                label=f'Threshold = {THRESHOLD}')
axes[0].set_xlabel('Recall (Sensitivity)', fontsize=11)
axes[0].set_ylabel('Precision', fontsize=11)
axes[0].set_title('Precision-Recall Curve', fontweight='bold')
axes[0].legend()
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# ROC Curve
fpr, tpr, roc_thresholds = roc_curve(y_holdout, y_prob)
axes[1].plot(fpr, tpr, color='#2E7D32', lw=2.5, label=f'AUC = {auc_roc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random classifier')
idx_roc = np.argmin(np.abs(roc_thresholds - THRESHOLD))
axes[1].scatter(fpr[idx_roc], tpr[idx_roc], s=150, color='#E53935', zorder=5,
                label=f'Threshold = {THRESHOLD}')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=11)
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('uplift_performance_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_performance_curves.png')

## 5. SHAP Feature Importance

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_holdout)

# Summary plot (beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_holdout, feature_names=feature_cols,
                  show=False, plot_size=None)
plt.title('UpLift — SHAP Feature Importance\n(Why the model flags equipment as at-risk)',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('uplift_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_shap_summary.png')

In [ ]:
# Save SHAP values as CSV for documentation
shap_df = pd.DataFrame(shap_values, columns=feature_cols)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
mean_abs_shap_df = mean_abs_shap.reset_index()
mean_abs_shap_df.columns = ['feature', 'mean_abs_shap']
mean_abs_shap_df.to_csv('uplift_shap_importance.csv', index=False)

print('Saved: uplift_shap_importance.csv')
print('\nTop features by mean |SHAP|:')
print(mean_abs_shap_df.to_string(index=False))

## 6. Operational Output: Ranked Equipment Risk List

In [ ]:
df_orig = pd.read_csv('uplift_equipment_data.csv')

# Apply model to full dataset for demonstration
df_model_input = df_orig.drop(columns=['equipment_id', 'outage_within_30_days'])
cat_cols = ['equipment_type', 'agency', 'manufacturer']
encoders = joblib.load('uplift_encoders.pkl')
for col in cat_cols:
    df_model_input[col] = encoders[col].transform(df_model_input[col])

full_probs = model.predict_proba(df_model_input.values)[:, 1]

risk_list = df_orig[['equipment_id', 'equipment_type', 'agency', 'equipment_age_years',
                      'unplanned_outages_12mo', 'days_since_last_maintenance']].copy()
risk_list['risk_score'] = full_probs
risk_list['risk_flag'] = (full_probs >= THRESHOLD).astype(int)
risk_list['priority_tier'] = pd.cut(
    full_probs,
    bins=[0, 0.30, 0.60, 0.80, 1.0],
    labels=['Monitor', 'Elevated', 'High', 'Critical']
)
risk_list = risk_list.sort_values('risk_score', ascending=False).reset_index(drop=True)
risk_list.to_csv('uplift_risk_scores.csv', index=False)

print('Saved: uplift_risk_scores.csv')
print(f'\nEquipment flagged for proactive service: {risk_list["risk_flag"].sum()}')
print(f'\nPriority tier breakdown:')
print(risk_list['priority_tier'].value_counts())
print(f'\nTop 10 highest-risk units:')
risk_list.head(10)

## Summary

| Output File | What It Is |
|---|---|
| `uplift_xgboost_model.pkl` | Trained model — ready for inference |
| `uplift_confusion_matrix.png` | Visualizes false negatives vs false positives |
| `uplift_performance_curves.png` | PR curve + ROC curve with operating threshold marked |
| `uplift_shap_summary.png` | Feature importance — explainable to transit agency staff |
| `uplift_shap_importance.csv` | SHAP values in tabular form |
| `uplift_risk_scores.csv` | Ranked equipment list — the operational deliverable |

---

**UpLift is designed to be handed to a maintenance scheduler.** The risk score is the output. Everything else is how we earned the right to trust it.

*Developed through the Equitech Futures Civic Tech Institute, CTI 2026 · Philadelphia, PA*